In [1]:
import os 
import sys
import pandas as pd
import traceback
import json

In [94]:
from dotenv import load_dotenv

load_dotenv(r"D:\Data Analysis\DeepLearning\DeepLearning_From_Scratch\huggingfacewithlangchain\mcqgen\.env")

True

In [98]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

In [101]:
from dotenv import dotenv_values

config = dotenv_values(
    r"D:\Data Analysis\DeepLearning\DeepLearning_From_Scratch\huggingfacewithlangchain\mcqgen\.env"
)

api_key = config["GROQ_KEY"]

In [107]:
# llm = ChatOpenAI(openai_api_key=api_key,model_name="gpt-4",temperature=0.5)
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_KEY"),
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

In [114]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C8E64591D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C8E6459450>, model_name='llama-3.3-70b-versatile', temperature=0.5, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [115]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.callbacks.manager import get_openai_callback
import PyPDF2

In [116]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [117]:
"""
number=5 
subject="data science"
tone="simple"
"""

'\nnumber=5 \nsubject="data science"\ntone="simple"\n'

In [118]:
TEMPLATE="""
Text:{text}
Act as an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [119]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [120]:
quiz_generation_prompt

PromptTemplate(input_variables=['number', 'response_json', 'subject', 'text', 'tone'], input_types={}, partial_variables={}, template='\nText:{text}\nAct as an expert MCQ maker. Given the above text, it is your job to create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. \nMake sure the questions are not repeated and check all the questions to be conforming the text as well.\nMake sure to format your response like  RESPONSE_JSON below  and use it as a guide. Ensure to make {number} MCQs\n### RESPONSE_JSON\n{response_json}\n\n')

In [121]:
quiz_chain = quiz_generation_prompt | llm

In [122]:
quiz_chain

PromptTemplate(input_variables=['number', 'response_json', 'subject', 'text', 'tone'], input_types={}, partial_variables={}, template='\nText:{text}\nAct as an expert MCQ maker. Given the above text, it is your job to create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. \nMake sure the questions are not repeated and check all the questions to be conforming the text as well.\nMake sure to format your response like  RESPONSE_JSON below  and use it as a guide. Ensure to make {number} MCQs\n### RESPONSE_JSON\n{response_json}\n\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 

In [123]:
TEMPLATE="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [124]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE)

In [125]:
review_chain = quiz_evaluation_prompt | llm

In [138]:
## combining the chains together
from langchain_core.runnables import RunnableLambda

generate_evaluate_chain = (
    quiz_chain
    | RunnableLambda(
        lambda quiz: {
            "quiz": quiz.content,
            "review": review_chain.invoke({
                "quiz": quiz.content,
                "subject": SUBJECT
            }).content
        }
    )
)

In [127]:
generate_evaluate_chain

PromptTemplate(input_variables=['number', 'response_json', 'subject', 'text', 'tone'], input_types={}, partial_variables={}, template='\nText:{text}\nAct as an expert MCQ maker. Given the above text, it is your job to create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. \nMake sure the questions are not repeated and check all the questions to be conforming the text as well.\nMake sure to format your response like  RESPONSE_JSON below  and use it as a guide. Ensure to make {number} MCQs\n### RESPONSE_JSON\n{response_json}\n\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 

In [128]:
filepath = r"D:\Data Analysis\DeepLearning\DeepLearning_From_Scratch\huggingfacewithlangchain\mcqgen\data.txt"

In [129]:
filepath

'D:\\Data Analysis\\DeepLearning\\DeepLearning_From_Scratch\\huggingfacewithlangchain\\mcqgen\\data.txt'

In [130]:
with open(filepath, "r") as f:
    text = f.read()

In [131]:
print(text)

Data science is an interdisciplinary academic field[1] that uses statistics, scientific computing, scientific methods, processes, algorithms and systems to extract or extrapolate knowledge and insights from noisy, structured, and unstructured data.[2]

Data science also integrates domain knowledge from the underlying application domain (e.g., natural sciences, information technology, and medicine).[3] Data science is multifaceted and can be described as a science, a research paradigm, a research method, a discipline, a workflow, and a profession.[4]

Data science is a "concept to unify statistics, data analysis, informatics, and their related methods" to "understand and analyze actual phenomena" with data.[5] It uses techniques and theories drawn from many fields within the context of mathematics, statistics, computer science, information science, and domain knowledge.[6] However, data science is different from computer science and information science. Turing Award winner Jim Gray imag

In [132]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [133]:
NUMBER=5 
SUBJECT="data science"
TONE="simple"

In [ ]:
# # track token usage in of Openai calls
# with get_openai_callback() as cb:
#     response = generate_evaluate_chain.invoke(
#         {
#             "text": text,
#             "number": NUMBER,
#             "subject": SUBJECT,
#             "tone": TONE,
#             "response_json": json.dumps(RESPONSE_JSON)
#         }
#     )

KeyError: "Input to PromptTemplate is missing variables {'subject'}.  Expected: ['quiz', 'subject'] Received: ['quiz']\nNote: if you intended {subject} to be part of the string and not a variable, please escape it with double curly braces like: '{{subject}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "

In [139]:
response = generate_evaluate_chain.invoke(
    {
        "text": text,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON)
    }
)

In [83]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:0
Prompt Tokens:0
Completion Tokens:0
Total Cost:0.0


In [140]:
quiz = response.get("quiz")

In [143]:
quiz=json.loads(quiz)

In [144]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [145]:
df=pd.DataFrame(quiz_table_data)

In [146]:
df

,MCQ,Choices,Correct
0,What is data science?,a: A field that only uses statistics | b: An i...,b
1,What is the role of a data scientist?,a: To only create programming code | b: To onl...,c
2,What are the emerging foundational professiona...,"a: Database management, statistics, and machin...",b
3,What is data science considered as?,a: Only a science | b: Only a research paradig...,c
4,Who imagined data science as a 'fourth paradig...,a: Nathan Yau | b: Ben Fry | c: Jim Gray | d: ...,c


In [147]:
df.to_csv("Data_Science_Quiz.csv",index=False)